# Bank private-credit exposure — full pipeline, source to result

This notebook reproduces the entire analysis behind the article, starting from the raw
data source. It runs in two halves:

1. **Acquire** — pull quarterly bank Call Report data from the FDIC's public API and
   assemble the panel. This is what `analysis/pull_call_reports.py` does; here it is inline
   so every field and filter is visible.
2. **Analyze** — aggregate that panel into the market total and the per-bank concentration
   metric. This is what `analysis/aggregate_ndfi.py` does.

Run cells top to bottom (Shift+Enter). Requires internet for the API calls in Part 1.

## The data source

**FDIC BankFind Suite — Financials API** (`https://api.fdic.gov/banks/financials`), free,
no key. It serves the quarterly **Call Report** (regulatory filing FFIEC 031/041) for every
FDIC-insured bank, as JSON.

**Reporting basis:** the *bank* (insured depository institution) level — each chartered bank
files its own report. This is **not** the holding-company level (that is a separate filing,
the FR Y-9C). We rely on this basis throughout, and later roll banks up to their parent by
name.

The call below requests one quarter. Each institution comes back as a record; here is one
bank's relevant fields, raw from the API.

In [ ]:
import json, urllib.parse, urllib.request

API = "https://api.fdic.gov/banks/financials"
FIELDS = ["CERT", "NAME", "NAMEHCR", "REPDTE",
          "ASSET", "LNLSNET", "LNNDEPC", "RBCT1J"]

def fdic_pull(repdte):
    query = urllib.parse.urlencode({
        "filters": f"REPDTE:{repdte}",
        "fields": ",".join(FIELDS),
        "limit": 10000, "format": "json",
    })
    with urllib.request.urlopen(f"{API}?{query}", timeout=90) as resp:
        return [d["data"] for d in json.load(resp)["data"]]

records = fdic_pull("20260331")           # 2026-Q1
print(f"{len(records)} institutions returned for 2026-Q1")

# one bank's raw record (Truist Bank, FDIC cert 9846)
next(r for r in records if r.get("CERT") == 9846)

## The fields, and what each one is

All dollar figures are in **thousands** ($000s) as filed. Field codes are the FDIC's.

| Field | What it is | From the Call Report |
|---|---|---|
| `CERT` | FDIC certificate no. — unique bank ID | (identifier) |
| `NAME` | Bank's legal name | (identifier) |
| `NAMEHCR` | Name of the top-tier **holding company** | (identifier) |
| `REPDTE` | Reporting date (quarter-end) | (identifier) |
| `ASSET` | Total assets | Schedule RC |
| `LNLSNET` | Net loans and leases | Schedule RC-C |
| `LNNDEPC` | **Loans to non-depository financial institutions (NDFI)** — the private-credit proxy | Schedule RC-C |
| `RBCT1J` | **Tier 1 (core) capital** — the loss-absorbing base | Schedule RC-R |

`LNNDEPC` is a newer line item: banks over $10B in assets have reported it only since
**2024-Q4**, which is why the panel starts there.

## Build the panel

Pull every quarter since the NDFI line began, keep only banks that actually report it
(the >$10B filers), and stack into one table. Dollar columns are converted to numbers.

This is exactly `analysis/pull_call_reports.py`, which also writes the result to
`analysis/outputs/call_report_ndfi_panel.parquet` (the file the article reads).

In [ ]:
import pandas as pd

QUARTERS = ["20241231", "20250331", "20250630", "20250930", "20251231", "20260331"]

rows = []
for q in QUARTERS:
    recs = [r for r in fdic_pull(q) if r.get("LNNDEPC") not in (None, "")]
    print(f"  {q}: {len(recs)} banks report an NDFI balance")
    rows.extend(recs)

panel = pd.DataFrame(rows)
panel["quarter"] = pd.to_datetime(panel["REPDTE"].astype(str), format="%Y%m%d")
for col in ["ASSET", "LNLSNET", "LNNDEPC", "RBCT1J"]:
    panel[col] = pd.to_numeric(panel[col], errors="coerce")

print(f"\npanel: {len(panel)} rows across {panel['quarter'].nunique()} quarters")
panel.head()

---
## Part 2 — Analyze

Everything below operates on the `panel` built above. The remaining steps are
`analysis/aggregate_ndfi.py`.

### 1. Isolate the latest quarter

Subset the panel to the most recent reporting date. `LNNDEPC` is the NDFI balance,
`RBCT1J` is Tier 1 capital — both per bank, in $thousands, shown here in $bn.

In [ ]:
latest = panel["quarter"].max()
cur = panel[panel["quarter"] == latest].copy()
print("latest quarter:", latest.date(), "| banks reporting NDFI:", len(cur))

view = cur[["NAME", "NAMEHCR", "LNNDEPC", "RBCT1J", "ASSET"]].copy()
view[["LNNDEPC","RBCT1J","ASSET"]] = (view[["LNNDEPC","RBCT1J","ASSET"]] / 1e6).round(1)
view.sort_values("LNNDEPC", ascending=False).head(8)

### 2. The market total

**Calculation:** sum `LNNDEPC` across every reporting bank in the quarter. That single
sum is the "$1.5 trillion market" headline. Top-10 share = the ten largest `LNNDEPC`
balances divided by the total.

In [ ]:
total_bn = cur["LNNDEPC"].sum() / 1e6
top10 = cur.nlargest(10, "LNNDEPC")["LNNDEPC"].sum() / 1e6
print(f"Total drawn NDFI:  ${total_bn:,.0f}B  (${total_bn/1000:.2f}T)")
print(f"Top-10 banks hold: {top10/total_bn:.0%} of it")

### 3. Why a rollup is needed — subsidiaries

`LNNDEPC` is reported per *bank*, but large firms operate several banks. Morgan Stanley
files two separate Call Reports (`NAME` differs; `NAMEHCR` is the same parent). Left as-is
they'd be two points with two wrong ratios.

In [ ]:
ms = cur.loc[cur["NAMEHCR"].str.contains("MORGAN STANLEY"),
             ["NAME", "NAMEHCR", "LNNDEPC", "RBCT1J"]].copy()
ms[["LNNDEPC","RBCT1J"]] = (ms[["LNNDEPC","RBCT1J"]] / 1e6).round(1)
ms

### 4. Roll subsidiaries up to the holding company

**Calculation:** group by `NAMEHCR` (the parent) and sum `LNNDEPC`, `RBCT1J`, `ASSET`,
`LNLSNET`. Now each firm is one row, on a consistent bank-consolidated basis.

In [ ]:
grp = (cur.assign(entity=cur["NAMEHCR"].str.strip().replace("", pd.NA).fillna(cur["NAME"]))
          .groupby("entity")
          .agg(ndfi=("LNNDEPC","sum"), tier1=("RBCT1J","sum"),
               assets=("ASSET","sum"), loans=("LNLSNET","sum"))
          .reset_index())

ms_holdco = grp[grp["entity"].str.contains("MORGAN STANLEY")]
(ms_holdco[["ndfi","tier1"]] / 1e6).round(1)

### 5. The concentration metric

**Calculation:** `ndfi_to_tier1 = LNNDEPC ÷ RBCT1J` per holding company — the NDFI book as
a share of Tier 1 capital. This is the single number the bubble chart's vertical axis and
the ranking are built on. A value over 100% means the private-credit book exceeds the
bank's entire core capital.

In [ ]:
grp["ndfi_to_tier1"] = grp["ndfi"] / grp["tier1"]
grp["ndfi_bn"] = grp["ndfi"] / 1e6

ranking = (grp[grp["ndfi_bn"] >= 10]
           .sort_values("ndfi_to_tier1", ascending=False)
           [["entity", "ndfi_bn", "ndfi_to_tier1"]]
           .head(15).reset_index(drop=True))
ranking["ndfi_bn"] = ranking["ndfi_bn"].round(1)
ranking["ndfi_to_tier1"] = (ranking["ndfi_to_tier1"]*100).round(0).astype(int).astype(str) + "%"
ranking

### 6. The bubble chart

A rough matplotlib version of the article's Observable Plot figure, from the same `grp`
table. Horizontal: `ndfi_bn` (log). Vertical: `ndfi_to_tier1`. Bubble area: total `assets`.
The dashed line marks 100% of Tier 1 capital.

In [ ]:
import matplotlib.pyplot as plt

g = grp[grp["ndfi_bn"] >= 1].copy()
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(g["ndfi_bn"], g["ndfi_to_tier1"], s=g["assets"]/1e4,
           alpha=0.4, color="#8CA0B4", edgecolor="#61758A")
ax.set_xscale("log")
ax.axhline(1.0, ls="--", color="#B4AD9E", lw=1)
ax.set_xlabel("NDFI loans, $bn (log)")
ax.set_ylabel("NDFI / Tier 1 capital")
ax.set_title("Scale is not concentration")
for name in ["GOLDMAN SACHS GROUP INC THE", "FIRST CITIZENS BANCSHARES INC", "TRUIST FINANCIAL CORP"]:
    row = g[g["entity"] == name]
    if len(row):
        ax.annotate(name.title()[:16], (row["ndfi_bn"].iloc[0], row["ndfi_to_tier1"].iloc[0]), fontsize=8)
plt.show()

### 7. Variations

Each of these changes one input above and re-runs downstream:
- **Different quarter:** set `latest` to an earlier value from `panel['quarter'].unique()` and re-run from step 1 to see the market total over time.
- **Different denominator:** `ndfi / loans` (`LNLSNET`) instead of `ndfi / tier1` — exposure as a share of the total loan book rather than of capital.
- **Single firm:** `cur[cur['NAMEHCR'].str.contains('WELLS')]` to inspect one company's subsidiaries.
- **Rank by dollars** instead of concentration to contrast the two lenses.

In [ ]:
# NDFI as a share of total loans (LNLSNET), by holding company
grp["ndfi_to_loans"] = grp["ndfi"] / grp["loans"]
(grp[grp["ndfi_bn"] >= 10]
   .sort_values("ndfi_to_loans", ascending=False)
   [["entity", "ndfi_bn", "ndfi_to_loans"]]
   .head(10))

---
## Part 3 — The aggregate market: growth, concentration, and the exposure you can't see

Parts 1–2 built the panel and computed the latest snapshot. This part steps back to the
*market shape* — the scene an outsider sets before zooming into any single bank: how fast
NDFI lending is growing, how concentrated it is, and why the drawn number understates the
true exposure. Everything still flows from the one Call Report field, `LNNDEPC`.

### Growth trajectory

Sum `LNNDEPC` by quarter. The NDFI line only begins 2024-Q4, so the series is short — and
the earliest quarters are muddied by *reclassification* (loans moved **into** the NDFI
bucket from other categories), so early "growth" is partly an accounting artifact. Read the
later quarters for the genuine trend.

In [ ]:
by_q = (panel.groupby("quarter")
              .agg(ndfi=("LNNDEPC", "sum"), loans=("LNLSNET", "sum")))
by_q["ndfi_bn"] = by_q["ndfi"] / 1e6
by_q["loans_bn"] = by_q["loans"] / 1e6
by_q["ndfi_qoq"] = by_q["ndfi_bn"].pct_change()
by_q[["ndfi_bn", "ndfi_qoq"]].round(3)

### Is NDFI growing faster than the rest of the loan book?

Compare NDFI growth to total-loan (`LNLSNET`) growth across the panel. Whalen's headline is
NDFI growing ~5x *other loan categories*; we only pulled the total loan line, not every
category, so this is NDFI vs. the whole book — directional, not the exact 5x.

In [ ]:
first, last = by_q.iloc[0], by_q.iloc[-1]
ndfi_growth = last.ndfi_bn / first.ndfi_bn - 1
loan_growth = last.loans_bn / first.loans_bn - 1
print(f"NDFI:         {ndfi_growth:+.1%}  over the panel")
print(f"Total loans:  {loan_growth:+.1%}  over the panel")
print("(NDFI growth is inflated by early reclassification — treat as an upper bound)")

### Concentration — a big-bank story

Sum the latest quarter by size tier. The takeaway that lets an outsider skip the ~4,000
small banks entirely: the giants *are* the market.

In [ ]:
tot = cur["LNNDEPC"].sum()
top10 = cur.nlargest(10, "LNNDEPC")["LNNDEPC"].sum()
big = cur.loc[cur["ASSET"] >= 100_000_000, "LNNDEPC"].sum()  # >$100B banks
print(f"Reporting banks:   {len(cur)}")
print(f"Top-10 banks hold: {top10/tot:.0%} of all NDFI")
print(f"Banks > $100B:     {big/tot:.0%}")

### The number you can't see — drawn vs. committed

Everything above is **drawn**: `LNNDEPC` is an *outstanding balance* (Schedule RC-C). The
larger risk is the **committed** line — the undrawn amount a bank has promised, which
stressed funds can draw all at once (*exposure at default*, a Basel-I idea). That figure
lives on a different schedule (**RC-L**, unused commitments) and isn't a clean per-NDFI
field, so estimating it takes inference — which is why the widely-cited ~$3T EAD number is
an **estimate**, not a sum. Illustratively, at a blended utilization (drawn ÷ committed)
the drawn book implies:

In [ ]:
drawn_bn = cur["LNNDEPC"].sum() / 1e6
print(f"drawn (clean sum):          ${drawn_bn/1000:.2f}T")
for util in (0.56, 0.47):
    print(f"implied committed @ {util:.0%}:   ${drawn_bn/util/1000:.1f}T  (illustrative)")

### Beyond the reporting banks

Only banks over $10B report NDFI directly, so this panel is ~80 firms — but they are
essentially the whole thing: banks over $100B alone are ~98% of it, and the FDIC's
*Quarterly Banking Profile* aggregate (all ~4,400 banks) lands within a rounding error of
our sum. What this data can **never** see is the non-bank market itself — the ~$2T of loans
the funds hold *directly*, which lives in fund filings (BDC 10-Ks, Form PF), not Call
Reports. Three concentric numbers, each less verifiable than the last:

| Number | Meaning | Source |
| --- | --- | --- |
| **~$1.5T** | bank NDFI **drawn** | one Call Report field (this notebook) |
| **~$3T** | bank NDFI **committed** (EAD) | Call Report + a utilization/CCF assumption |
| **~$2T** | the private-credit **market** | fund filings — outside the banking data entirely |

The number you can trust most is the smallest one.